# عامل ترجمة الخزانة المؤقت: Kaggle CPU → Cloudflare

يستخدم النموذج الرسمي المحفوظ والربط الخاص نفسيهما. هذا مسار مؤقت بعد نفاد حصة GPU، وسيكون أبطأ. يواصل من عداد Cloudflare للغات الـ22، ولا يعيد تنزيل النموذج.


In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import hashlib, json, os, signal, socket, subprocess, tarfile, threading, time
import urllib.error, urllib.request

MODEL_NAME = "Hy-MT2-7B-Q4_K_M.gguf"
MODEL_SIZE = 4_624_648_896
MODEL_SHA = "9f96256500f3fc1ab4d64336b58f52a949a95ad7516b0c229476eef782f9f77b"
RELEASE = "b11580"
RUNTIME = {
    "llama-b11580-bin-ubuntu-x64.tar.gz": (17_723_240, "de67caf142984e7ba2fb2b5396d6ea40f2946a512650d1f7e9d38ea1a6e5fac7"),
}
LOCAL_LANGUAGES = {
    "en":"English", "fr":"French", "tr":"Turkish", "ur":"Urdu", "ug":"Uyghur",
    "fa":"Persian", "hi":"Hindi", "id":"Indonesian", "ms":"Malay", "bn":"Bengali",
    "ru":"Russian", "uk":"Ukrainian", "de":"German", "es":"Spanish", "pt":"Portuguese",
    "it":"Italian", "nl":"Dutch", "pl":"Polish", "kk":"Kazakh", "zh":"Chinese",
    "ja":"Japanese", "ko":"Korean",
}
SOURCES = {}
WORK = Path("/kaggle/working/khzanah-cloudflare")
WORK.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    size = path.stat().st_size
    digest = hashlib.sha256()
    done = 0
    last = 0.0
    with path.open("rb") as stream:
        while chunk := stream.read(8 * 1024 * 1024):
            digest.update(chunk)
            done += len(chunk)
            now = time.monotonic()
            if now - last >= 5 or done == size:
                print(f"فحص {path.name}: {done / size:.1%}", flush=True)
                last = now
    return digest.hexdigest()

def verify(path, expected_size, expected_sha):
    if not path.is_file() or path.stat().st_size != expected_size:
        raise RuntimeError(f"حجم الملف غير مطابق: {path.name}")
    if sha256_file(path) != expected_sha:
        raise RuntimeError(f"بصمة الملف غير مطابقة: {path.name}")
    return path

def request_json(url, timeout=45):
    request = urllib.request.Request(url, headers={
        "User-Agent": "Mozilla/5.0 KhzanahKaggle/1.0", "Accept": "application/json"
    })
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)

def wait_health(url, process, log_path, timeout=600):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            tail = log_path.read_text(encoding="utf-8", errors="replace")[-4000:]
            raise RuntimeError("توقف llama-server أثناء التحميل:\n" + tail)
        try:
            if request_json(url + "/health", 3).get("status") == "ok":
                return
        except Exception:
            time.sleep(3)
    raise RuntimeError("انتهت مهلة تحميل النموذج بعد عشر دقائق")

def print_progress(worker_url):
    data = request_json(worker_url.rstrip("/") + "/api/translate/automatic")
    print("\nتقدم اللغات المحلية الـ22 — محفوظ في Cloudflare", flush=True)
    print("code language              ready remaining percent", flush=True)
    for code, name in LOCAL_LANGUAGES.items():
        row = data.get("byLanguage", {}).get(code, {})
        ready, remaining = int(row.get("ready", 0)), int(row.get("remaining", 0))
        percent = 100 * ready / max(1, ready + remaining)
        marker = "  <== النشطة" if code == data.get("localActiveLanguage") else ""
        print(f"{code:>4} {name:<20} {ready:>6,} {remaining:>9,} {percent:>6.2f}%{marker}", flush=True)
    active = data.get("localActiveLanguage") or "-"
    print(f"اللغة المحلية النشطة: {active} | المصادر: {data.get('sources', 0):,}\n", flush=True)

def progress_loop(worker_url, stop):
    while not stop.is_set():
        try:
            print_progress(worker_url)
        except Exception as error:
            print("تعذر تحديث جدول التقدم؛ سيعاد تلقائيًا:", str(error)[:160], flush=True)
        stop.wait(300)

assert Path("/kaggle/input").is_dir(), "يشغّل هذا الدفتر داخل Kaggle فقط"
secrets = UserSecretsClient()
bridge_tokens = sorted(Path("/kaggle/input").rglob("khzanah-agent-token.txt"))
if len(bridge_tokens) == 1:
    token = bridge_tokens[0].read_text(encoding="utf-8").strip()
else:
    try:
        token = secrets.get_secret("CLOUDFLARE_AGENT_TOKEN")
    except Exception as error:
        raise RuntimeError("مصدر ربط Cloudflare الخاص غير مرفق بالدفتر") from error
try:
    worker_url = secrets.get_secret("CLOUDFLARE_WORKER_URL")
except Exception:
    worker_url = "https://khzanah.com"
if not isinstance(token, str) or len(token) < 32:
    raise RuntimeError("سر CLOUDFLARE_AGENT_TOKEN غير موجود أو غير صالح")
if not worker_url.startswith("https://"):
    raise RuntimeError("عنوان Cloudflare يجب أن يبدأ بـ https://")

models = sorted(path for path in Path("/kaggle/input").rglob(MODEL_NAME) if path.is_file())
if len(models) != 1:
    raise RuntimeError("مصدر التخزين المرفق يجب أن يحتوي نموذجًا واحدًا فقط")
model = verify(models[0], MODEL_SIZE, MODEL_SHA)
cache = model.parent

def cpu_runtime_archive(name, size, digest):
    cache_dir = Path("/kaggle/working/khzanah-cpu-runtime")
    cache_dir.mkdir(parents=True, exist_ok=True)
    destination = cache_dir / name
    if destination.exists():
        return verify(destination, size, digest)
    url = f"https://github.com/ggml-org/llama.cpp/releases/download/{RELEASE}/{name}"
    temporary = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    for attempt in range(1, 4):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "KhzanahKaggleCPU/1.0"})
            with urllib.request.urlopen(request, timeout=90) as response, temporary.open("wb") as output:
                if response.status != 200 or int(response.headers.get("Content-Length", size)) != size:
                    raise RuntimeError("حجم تنزيل llama.cpp غير مطابق")
                while chunk := response.read(1024 * 1024):
                    output.write(chunk)
                    print(f"llama.cpp CPU: {output.tell() / size:.0%}", flush=True)
            verify(temporary, size, digest)
            temporary.replace(destination)
            return destination
        except Exception as error:
            last_error = error
            print(f"تعذر تنزيل runtime؛ محاولة {attempt}/3: {type(error).__name__}", flush=True)
            time.sleep(min(10, attempt * 2))
    raise RuntimeError("تعذر تنزيل llama.cpp CPU الرسمي بعد 3 محاولات") from last_error


runtime_dir = Path("/tmp/khzanah-llama-b11580-cpu")
runtime_dir.mkdir(parents=True, exist_ok=True)
for name, (size, digest) in RUNTIME.items():
    archive = cpu_runtime_archive(name, size, digest)
    with tarfile.open(archive, "r:gz") as bundle:
        if not hasattr(tarfile, "data_filter"):
            raise RuntimeError("إصدار Python لا يدعم الاستخراج الآمن")
        bundle.extractall(runtime_dir, filter="data")

servers = sorted(path for path in runtime_dir.rglob("llama-server") if path.is_file())
if len(servers) != 1:
    raise RuntimeError("لم يُعثر على llama-server واحد")
llama = servers[0]
llama.chmod(0o755)

for relative, content in SOURCES.items():
    destination = WORK / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(content, encoding="utf-8")

if (os.cpu_count() or 0) < 2:
    raise RuntimeError("دفتر CPU لا يملك نواتين على الأقل")
print(f"CPU fallback: {os.cpu_count()} logical cores; one translation lane", flush=True)

server_env = os.environ.copy()
library_dirs = sorted({str(path.parent) for path in runtime_dir.rglob("*.so*")})
server_env["LD_LIBRARY_PATH"] = ":".join(library_dirs + [server_env.get("LD_LIBRARY_PATH", "")])
port = 8080
server_url = f"http://127.0.0.1:{port}"
server_log = WORK / "llama-server.log"
server_stream = server_log.open("w", encoding="utf-8")
server = subprocess.Popen([
    str(llama), "-m", str(model), "-ngl", "0",
    "-c", "2048", "--parallel", "1", "--jinja", "--host", "127.0.0.1",
    "--port", str(port), "-b", "128", "-ub", "64", "-t", str(min(4, os.cpu_count() or 2)),
], env=server_env, stdout=server_stream, stderr=subprocess.STDOUT, start_new_session=True)

agent = None
monitor_stop = threading.Event()
monitor = None
started = time.time()
try:
    wait_health(server_url, server, server_log)
    print("llama.cpp CPU healthy; Cloudflare agent starting", flush=True)
    print_progress(worker_url)
    monitor = threading.Thread(target=progress_loop, args=(worker_url, monitor_stop), daemon=True)
    monitor.start()

    agent_env = os.environ.copy()
    agent_env.update({
        "LOCAL_TRANSLATION_URL": worker_url,
        "LOCAL_TRANSLATION_AGENT_TOKEN": token,
        "LOCAL_TRANSLATION_CONCURRENCY": "1",
        "LOCAL_LLAMA_PORT": str(port),
    })
    token = None
    agent = subprocess.Popen(
        ["node", str(WORK / "local-agent/agent.mjs")], cwd=WORK, env=agent_env,
        start_new_session=True,
    )
    agent_env.pop("LOCAL_TRANSLATION_AGENT_TOKEN", None)
    print("بدأت ترجمة اللغة المحلية النشطة على CPU بمسار واحد. كل نتيجة تحفظ فورًا في Cloudflare.", flush=True)
    print("CPU continues until Kaggle stops the session or the queue completes.", flush=True)
    summary_url = worker_url.rstrip("/") + "/api/translate/summary"
    initial = request_json(summary_url)
    last_progress = (initial.get("language"), int(initial.get("ready", 0)))
    last_advance = time.monotonic()
    next_check = time.monotonic() + 60
    while True:
        if time.monotonic() >= next_check:
            next_check = time.monotonic() + 60
            try:
                snapshot = request_json(summary_url)
                current = (snapshot.get("language"), int(snapshot.get("ready", 0)))
                if current[0] != last_progress[0] or current[1] > last_progress[1]:
                    last_progress = current
                    last_advance = time.monotonic()
                elif time.monotonic() - last_advance > 12 * 60:
                    raise RuntimeError("No verified translation progress for 12 minutes; stopping CPU session")
            except (urllib.error.URLError, TimeoutError, ValueError) as error:
                print(f"Progress check unavailable: {type(error).__name__}", flush=True)
                if time.monotonic() - last_advance > 12 * 60:
                    raise RuntimeError("No verified Cloudflare progress for 12 minutes; stopping CPU session")
        code = agent.poll()
        if code is not None:
            if code != 0:
                raise RuntimeError(f"توقف عامل الترجمة برمز {code}")
            break
        time.sleep(15)
finally:
    monitor_stop.set()
    if agent and agent.poll() is None:
        agent.send_signal(signal.SIGINT)
        try:
            agent.wait(timeout=45)
        except subprocess.TimeoutExpired:
            agent.terminate()
            agent.wait(timeout=15)
    if server.poll() is None:
        server.send_signal(signal.SIGINT)
        try:
            server.wait(timeout=45)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=15)
    server_stream.close()
    summary = {
        "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(started)),
        "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "model_sha256": MODEL_SHA,
        "llama_release": RELEASE,
        "concurrency": 1,
        "checkpoint": "Cloudflare per accepted translation; leases resume automatically",
        "agent_exit_code": None if agent is None else agent.poll(),
    }
    (WORK / "session-summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    try:
        print_progress(worker_url)
    except Exception:
        pass
    print("انتهت جلسة Kaggle بأمان. التقدم محفوظ في Cloudflare ويمكن استئنافه بتشغيل الدفتر نفسه.", flush=True)
